In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys

In [18]:
excel_path = Path("/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx")

excel_file = pd.ExcelFile(excel_path)

print("Available sheets:")
for sheet in excel_file.sheet_names:
    print(sheet)

Available sheets:
README
data_dictionary
attrition_dataset
protected_attributes
attrition_scoring_2026
engagement_survey
performance_reviews
attendance_monthly_features
attendance_anomaly_labels
attendance_daily_examples
payroll_monthly_features
payroll_anomaly_labels
monthly_headcount_history
workforce_events
salary_training
job_descriptions
resumes
resume_job_labels
resume_bias_pairs
resume_protected_attrs
chatbot_nlu_training
chatbot_test_set
chatbot_responses
hr_faq
chatbot_dialogues
baseline_benchmarks
anomaly_baselines
sample_attrition_risk
sample_anomaly_results
sample_forecast
answer_key_planted_dirt
answer_key_notes


In [19]:
file_path=Path("/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx")

In [34]:
sample_anomaly_results=pd.read_excel(file_path,sheet_name="sample_anomaly_results")

In [35]:
sample_anomaly_results.shape

(405, 12)

In [36]:
sample_anomaly_results.columns.tolist()

['result_id',
 'source_dataset',
 'record_id',
 'employee_id',
 'period',
 'anomaly_score',
 'flagged',
 'reason_code',
 'reason_text',
 'money_at_risk',
 'model_version',
 'scored_at']

In [37]:
sample_anomaly_results.head()

,result_id,source_dataset,record_id,employee_id,period,anomaly_score,flagged,reason_code,reason_text,money_at_risk,model_version,scored_at
0,AN-0001,attendance_monthly_features,AF-03170,EMP-0356,2025-07-01,1.000,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
1,AN-0002,attendance_monthly_features,AF-05918,EMP-0585,2025-09-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
2,AN-0003,attendance_monthly_features,AF-06802,EMP-0646,2026-06-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
3,AN-0004,attendance_monthly_features,AF-02819,EMP-0318,2026-06-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
4,AN-0005,attendance_monthly_features,AF-03112,EMP-0347,2026-09-01,0.992,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00


In [38]:
sample_anomaly_results.dtypes

result_id                    str
source_dataset               str
record_id                    str
employee_id                  str
period            datetime64[us]
anomaly_score            float64
flagged                      str
reason_code                  str
reason_text                  str
money_at_risk            float64
model_version                str
scored_at         datetime64[us]
dtype: object

In [39]:
sample_anomaly_results["source_dataset"].value_counts()

source_dataset
attendance_monthly_features    221
payroll_monthly_features       184
Name: count, dtype: int64

In [40]:
sample_anomaly_results["reason_code"].value_counts()

reason_code
STATISTICAL_OUTLIER     145
PATTERN_SHIFT           125
NO_ATTENDANCE            50
LONG_UNPAID_ABSENCE      43
GROSS_ABOVE_NORM         13
NEGATIVE_NET_PAY          8
OVERTIME_ABOVE_CAP        8
ZERO_PAY                  5
GROSS_BELOW_EXPECTED      4
FREQUENT_LATE             3
PF_NOT_DEDUCTED           1
Name: count, dtype: int64

In [42]:
sample_anomaly_results["flagged"].value_counts()

flagged
Y    405
Name: count, dtype: int64

In [43]:
# payroll inspect 

In [44]:
payroll_sample = sample_anomaly_results[
    sample_anomaly_results["source_dataset"]
    == "payroll_monthly_features"
].copy()

print("Payroll sample shape:", payroll_sample.shape)

print("\nReason code summary:")
display(
    payroll_sample.groupby("reason_code").agg(
        record_count=("record_id", "count"),
        money_values=("money_at_risk", "count"),
        average_risk=("money_at_risk", "mean"),
        total_risk=("money_at_risk", "sum")
    ).sort_values("record_count", ascending=False)
)

print("\nExample payroll alerts:")
display(
    payroll_sample.groupby("reason_code", group_keys=False)
    .head(3)[[
        "record_id",
        "employee_id",
        "period",
        "anomaly_score",
        "reason_code",
        "reason_text",
        "money_at_risk"
    ]]
)

Payroll sample shape: (184, 12)

Reason code summary:


,record_count,money_values,average_risk,total_risk
reason_code,,,,
STATISTICAL_OUTLIER,145,145,134020.594897,19432986.26
GROSS_ABOVE_NORM,13,13,724455.713077,9417924.27
NEGATIVE_NET_PAY,8,8,113639.351250,909114.81
OVERTIME_ABOVE_CAP,8,8,34095.938750,272767.51
ZERO_PAY,5,5,86252.550000,431262.75
GROSS_BELOW_EXPECTED,4,4,231765.957500,927063.83
PF_NOT_DEDUCTED,1,1,30119.640000,30119.64



Example payroll alerts:


,record_id,employee_id,period,anomaly_score,reason_code,reason_text,money_at_risk
221,PI-005869,EMP-0503,2026-06-01,1.000,NEGATIVE_NET_PAY,Deductions exceed gross pay,347844.46
222,PI-003304,EMP-0304,2025-12-01,0.939,STATISTICAL_OUTLIER,Unusual combination of pay ratios versus all o...,2165286.90
223,PI-002595,EMP-0453,2025-10-01,0.936,STATISTICAL_OUTLIER,Unusual combination of pay ratios versus all o...,1835162.77
224,PI-002124,EMP-0331,2025-09-01,0.923,GROSS_ABOVE_NORM,Gross is 10.0x the monthly fixed pay,1413294.48
225,PI-003441,EMP-0533,2025-12-01,0.922,GROSS_ABOVE_NORM,Gross is 10.0x the monthly fixed pay,1329305.58
226,PI-000741,EMP-0578,2025-05-01,0.914,NEGATIVE_NET_PAY,Deductions exceed gross pay,33214.43
227,PI-003796,EMP-0459,2026-01-01,0.911,GROSS_ABOVE_NORM,Gross is 10.0x the monthly fixed pay,854779.35
228,PI-005556,EMP-0633,2026-05-01,0.910,NEGATIVE_NET_PAY,Deductions exceed gross pay,24669.69
234,PI-000998,EMP-0423,2025-06-01,0.895,STATISTICAL_OUTLIER,Unusual combination of pay ratios versus all o...,297820.58
251,PI-000281,EMP-0485,2025-04-01,0.838,OVERTIME_ABOVE_CAP,96 paid overtime hours exceeds the 50-hour cap,47557.75


In [45]:
payroll_data = pd.read_excel(
    excel_path,
    sheet_name="payroll_monthly_features"
)

sample_risk_check = payroll_sample.merge(
    payroll_data,
    left_on="record_id",
    right_on="payroll_item_id",
    how="left"
)

print(
    "Matched records:",
    sample_risk_check["payroll_item_id"].notna().sum(),
    "out of",
    len(payroll_sample)
)

columns_to_check = [
    "record_id",
    "reason_code",
    "money_at_risk",
    "monthly_fixed_gross",
    "payable_days",
    "lop_days",
    "overtime_hours_paid",
    "overtime_pay",
    "gross_pay",
    "pf_employee",
    "total_deductions",
    "net_pay"
]

display(
    sample_risk_check.groupby(
        "reason_code",
        group_keys=False
    ).head(2)[columns_to_check]
)

Matched records: 184 out of 184


,record_id,reason_code,money_at_risk,monthly_fixed_gross,payable_days,lop_days,overtime_hours_paid,overtime_pay,gross_pay,pf_employee,total_deductions,net_pay
0,PI-005869,NEGATIVE_NET_PAY,347844.46,201832.0,29.0,1.0,0.0,0.00,514200.66,1800.00,688444.91,-174244.25
1,PI-003304,STATISTICAL_OUTLIER,2165286.90,197044.0,31.0,0.0,0.0,0.00,2823640.50,1800.00,47789.65,2345744.00
2,PI-002595,STATISTICAL_OUTLIER,1835162.77,232431.0,31.0,0.0,0.0,0.00,2424310.00,1800.00,39288.47,2031425.30
3,PI-002124,GROSS_ABOVE_NORM,1413294.48,177578.0,30.0,0.0,0.0,0.00,1775780.00,1800.00,20545.28,1570327.20
4,PI-003441,GROSS_ABOVE_NORM,1329305.58,165489.0,31.0,0.0,0.0,0.00,1654890.00,1800.00,17788.38,1477006.20
5,PI-000741,NEGATIVE_NET_PAY,33214.43,27532.0,28.5,2.5,10.0,2647.31,27958.98,1214.98,32169.86,-4210.88
30,PI-000281,OVERTIME_ABOVE_CAP,47557.75,35593.0,30.0,0.0,96.0,43806.80,87942.12,1708.44,1908.44,86033.68
45,PI-003529,OVERTIME_ABOVE_CAP,41464.57,36011.0,31.0,0.0,102.0,47091.28,83102.28,1728.48,1928.48,81173.80
74,PI-001866,ZERO_PAY,37745.00,37945.0,31.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00
75,PI-004734,ZERO_PAY,56540.00,56040.0,31.0,0.0,0.0,0.00,0.00,1800.00,0.00,0.00


In [46]:
reason_definitions = {
    "negative_net_pay": {
        "reason_code": "NEGATIVE_NET_PAY",
        "reason_text": (
            "Total deductions exceed gross pay, "
            "resulting in negative net pay."
        )
    },

    "zero_net_pay": {
        "reason_code": "ZERO_PAY",
        "reason_text": (
            "The employee's recorded net pay is zero "
            "despite having payable days."
        )
    },

    "matching_payroll_record": {
        "reason_code": "DOUBLE_PAYMENT",
        "reason_text": (
            "A matching payment exists for the same employee, "
            "month and payroll run. Verify whether it is a duplicate."
        )
    },

    "iforest_anomaly": {
        "reason_code": "STATISTICAL_OUTLIER",
        "reason_text": (
            "The employee's payroll ratios are unusual "
            "compared with other payroll records."
        )
    }
}

for flag, details in reason_definitions.items():
    print(flag, "->", details["reason_code"])
    print("Reason:", details["reason_text"])
    print()

negative_net_pay -> NEGATIVE_NET_PAY
Reason: Total deductions exceed gross pay, resulting in negative net pay.

zero_net_pay -> ZERO_PAY
Reason: The employee's recorded net pay is zero despite having payable days.

matching_payroll_record -> DOUBLE_PAYMENT
Reason: A matching payment exists for the same employee, month and payroll run. Verify whether it is a duplicate.

iforest_anomaly -> STATISTICAL_OUTLIER
Reason: The employee's payroll ratios are unusual compared with other payroll records.



In [47]:
import pandas as pd
from pathlib import Path

export_path = Path(
    "/workspace/reports/payroll_features_task6.csv"
)

payroll_features = pd.read_csv(export_path)

print("Shape:", payroll_features.shape)

print("\nColumns:")
print(payroll_features.columns.tolist())

print("\nTotal flagged records:")
print(payroll_features["combined_anomaly"].sum())

print("\nIndividual flag counts:")
display(
    payroll_features[
        [
            "iforest_anomaly",
            "negative_net_pay",
            "zero_net_pay",
            "matching_payroll_record",
            "combined_anomaly"
        ]
    ].sum()
)

Shape: (7355, 34)

Columns:
['payroll_item_id', 'run_id', 'employee_id', 'month', 'department_id', 'role_id', 'job_level', 'overtime_eligible', 'monthly_fixed_gross', 'payable_days', 'lop_days', 'overtime_hours_paid', 'overtime_pay', 'variable_pay', 'variable_pay_type', 'gross_pay', 'pf_employee', 'esi_employee', 'professional_tax', 'tds', 'other_deductions', 'total_deductions', 'net_pay', 'gross_fixed_ratio', 'deduction_ratio', 'net_ratio', 'negative_net_pay', 'zero_net_pay', 'duplicate_payment', 'matching_payroll_record', 'iforest_anomaly', 'iforest_anomaly_score', 'rule_anomaly', 'combined_anomaly']

Total flagged records:
219

Individual flag counts:


iforest_anomaly            184
negative_net_pay             8
zero_net_pay                 5
matching_payroll_record     30
combined_anomaly           219
dtype: int64

In [48]:
print("Payroll features shape:", payroll_features.shape)

required_columns = [
    "payroll_item_id",
    "employee_id",
    "month",
    "iforest_anomaly",
    "negative_net_pay",
    "zero_net_pay",
    "matching_payroll_record",
    "combined_anomaly"
]

print("\nAvailable required columns:")
for column in required_columns:
    print(column, ":", column in payroll_features.columns)

print("\nFlag counts:")
display(
    payroll_features[[
        "iforest_anomaly",
        "negative_net_pay",
        "zero_net_pay",
        "matching_payroll_record",
        "combined_anomaly"
    ]].sum()
)

Payroll features shape: (7355, 34)

Available required columns:
payroll_item_id : True
employee_id : True
month : True
iforest_anomaly : True
negative_net_pay : True
zero_net_pay : True
matching_payroll_record : True
combined_anomaly : True

Flag counts:


iforest_anomaly            184
negative_net_pay             8
zero_net_pay                 5
matching_payroll_record     30
combined_anomaly           219
dtype: int64

In [49]:
print("Anomaly score statistics:")
display(
    payroll_features["iforest_anomaly_score"].describe()
)

print("\nDifferent flag combinations:")
display(
    payroll_features.groupby(
        [
            "iforest_anomaly",
            "negative_net_pay",
            "zero_net_pay",
            "matching_payroll_record"
        ]
    ).size().reset_index(name="count")
)

print("\nExamples of flagged payroll records:")

columns = [
    "payroll_item_id",
    "employee_id",
    "month",
    "monthly_fixed_gross",
    "payable_days",
    "lop_days",
    "overtime_pay",
    "variable_pay",
    "gross_pay",
    "total_deductions",
    "net_pay",
    "iforest_anomaly_score"
]

display(
    payroll_features[
        payroll_features["combined_anomaly"]
    ][columns].head(10)
)

Anomaly score statistics:


count    7355.000000
mean        0.198112
std         0.074038
min        -0.192363
25%         0.169128
50%         0.222774
75%         0.252078
max         0.268568
Name: iforest_anomaly_score, dtype: float64


Different flag combinations:


,iforest_anomaly,negative_net_pay,zero_net_pay,matching_payroll_record,count
0,False,False,False,False,7136
1,False,False,False,True,30
2,False,False,True,False,5
3,True,False,False,False,176
4,True,True,False,False,8



Examples of flagged payroll records:


,payroll_item_id,employee_id,month,monthly_fixed_gross,payable_days,lop_days,overtime_pay,variable_pay,gross_pay,total_deductions,net_pay,iforest_anomaly_score
16,PI-000017,EMP-0035,2025-04-01,420458.0,30.0,0.0,0.0,227047.32,647505.32,165671.66,481833.66,-0.039704
24,PI-000025,EMP-0049,2025-04-01,465375.0,29.0,1.0,0.0,362992.50,812855.00,218539.09,594315.91,-0.072310
27,PI-000028,EMP-0055,2025-04-01,238297.0,30.0,0.0,0.0,185871.66,424168.66,95990.62,328178.04,-0.019299
36,PI-000037,EMP-0072,2025-04-01,340979.0,30.0,0.0,0.0,184128.66,525107.66,127483.59,397624.07,-0.019618
52,PI-000053,EMP-0099,2025-04-01,633919.0,30.0,0.0,0.0,342316.26,976235.26,268235.40,707999.86,-0.061304
55,PI-000056,EMP-0107,2025-04-01,434589.0,30.0,0.0,0.0,234678.06,669267.06,172461.32,496805.74,-0.042392
59,PI-000060,EMP-0112,2025-04-01,332694.0,30.0,0.0,0.0,179654.76,512348.76,123502.82,388845.94,-0.016223
86,PI-000087,EMP-0161,2025-04-01,381400.0,28.5,1.5,0.0,205956.00,568286.00,142872.73,425413.27,-0.032496
99,PI-000100,EMP-0198,2025-04-01,470781.0,30.0,0.0,0.0,367209.18,837990.18,225102.93,612887.25,-0.072779
126,PI-000127,EMP-0249,2025-04-01,304247.0,29.0,1.0,0.0,164293.38,458398.82,107948.76,350450.06,-0.007142


In [50]:
#verify duplicate payments and zero pay records 

In [51]:
print("DUPLICATE PAYMENT EXAMPLES")

duplicate_rows = payroll_features[
    payroll_features["matching_payroll_record"]
].copy()

display(
    duplicate_rows[
        [
            "payroll_item_id",
            "employee_id",
            "month",
            "run_id",
            "gross_pay",
            "total_deductions",
            "net_pay"
        ]
    ].sort_values(
        ["employee_id", "month", "payroll_item_id"]
    ).head(12)
)

print(
    "\nRows with PI-D prefix:",
    duplicate_rows["payroll_item_id"].str.startswith("PI-D").sum()
)

print("\nZERO NET PAY EXAMPLES")

display(
    payroll_features[
        payroll_features["zero_net_pay"]
    ][
        [
            "payroll_item_id",
            "employee_id",
            "monthly_fixed_gross",
            "payable_days",
            "lop_days",
            "gross_pay",
            "net_pay"
        ]
    ]
)

print("\nNEGATIVE NET PAY EXAMPLES")

display(
    payroll_features[
        payroll_features["negative_net_pay"]
    ][
        [
            "payroll_item_id",
            "monthly_fixed_gross",
            "gross_pay",
            "total_deductions",
            "net_pay"
        ]
    ]
)

DUPLICATE PAYMENT EXAMPLES


,payroll_item_id,employee_id,month,run_id,gross_pay,total_deductions,net_pay
3975,PI-003970,EMP-0012,2026-02-01,PR-2026-02,53915.0,2000.00,51915.00
3976,PI-D00007,EMP-0012,2026-02-01,PR-2026-02,53915.0,2000.00,51915.00
5673,PI-005661,EMP-0092,2026-06-01,PR-2026-06,73903.0,2000.00,71903.00
5674,PI-D00015,EMP-0092,2026-06-01,PR-2026-06,73903.0,2000.00,71903.00
2416,PI-002412,EMP-0112,2025-10-01,PR-2025-10,342694.0,70570.53,272123.47
2417,PI-D00002,EMP-0112,2025-10-01,PR-2025-10,342694.0,70570.53,272123.47
4058,PI-004052,EMP-0179,2026-02-01,PR-2026-02,88641.0,2000.00,86641.00
4059,PI-D00001,EMP-0179,2026-02-01,PR-2026-02,88641.0,2000.00,86641.00
496,PI-000497,EMP-0218,2025-05-01,PR-2025-05,178808.0,20865.08,157942.92
497,PI-D00008,EMP-0218,2025-05-01,PR-2025-05,178808.0,20865.08,157942.92



Rows with PI-D prefix: 15

ZERO NET PAY EXAMPLES


,payroll_item_id,employee_id,monthly_fixed_gross,payable_days,lop_days,gross_pay,net_pay
1870,PI-001866,EMP-0540,37945.0,31.0,0.0,0.0,0.0
4744,PI-004734,EMP-0627,56040.0,31.0,0.0,0.0,0.0
4875,PI-004865,EMP-0142,139318.0,30.0,0.0,0.0,0.0
5058,PI-005048,EMP-0503,201832.0,29.0,1.0,0.0,0.0
5652,PI-005640,EMP-0047,51835.0,30.0,0.0,0.0,0.0



NEGATIVE NET PAY EXAMPLES


,payroll_item_id,monthly_fixed_gross,gross_pay,total_deductions,net_pay
492,PI-000493,196614.0,193442.81,237902.78,-44459.97
743,PI-000741,27532.0,27958.98,32169.86,-4210.88
3099,PI-003094,55709.0,55709.00,63279.90,-7570.90
5454,PI-005443,101865.0,101865.00,114051.50,-12186.50
5568,PI-005556,22014.0,23389.88,26785.59,-3395.71
5882,PI-005869,201832.0,514200.66,688444.91,-174244.25
6080,PI-006067,69857.0,69857.00,78842.70,-8985.70
6872,PI-006858,41208.0,41208.00,47328.80,-6120.80


In [52]:
duplicate_columns = [
    "employee_id",
    "month",
    "run_id",
    "gross_pay",
    "total_deductions",
    "net_pay"
]

duplicate_rows = payroll_features[
    payroll_features["matching_payroll_record"]
].copy()

duplicate_groups = (
    duplicate_rows.groupby(
        duplicate_columns,
        dropna=False
    )
    .agg(
        record_count=("payroll_item_id", "size"),
        payroll_ids=("payroll_item_id", list)
    )
    .reset_index()
)

duplicate_groups["pi_d_count"] = duplicate_groups[
    "payroll_ids"
].apply(
    lambda ids: sum(
        str(x).startswith("PI-D") for x in ids
    )
)

print("Duplicate groups:", len(duplicate_groups))
print("Total matching records:", len(duplicate_rows))

print("\nRecords per group:")
display(duplicate_groups["record_count"].value_counts())

print("\nPI-D records per group:")
display(duplicate_groups["pi_d_count"].value_counts())

display(duplicate_groups.head(10))

Duplicate groups: 15
Total matching records: 30

Records per group:


record_count
2    15
Name: count, dtype: int64


PI-D records per group:


pi_d_count
1    15
Name: count, dtype: int64

,employee_id,month,run_id,gross_pay,total_deductions,net_pay,record_count,payroll_ids,pi_d_count
0,EMP-0012,2026-02-01,PR-2026-02,53915.00,2000.00,51915.00,2,"[PI-003970, PI-D00007]",1
1,EMP-0092,2026-06-01,PR-2026-06,73903.00,2000.00,71903.00,2,"[PI-005661, PI-D00015]",1
2,EMP-0112,2025-10-01,PR-2025-10,342694.00,70570.53,272123.47,2,"[PI-002412, PI-D00002]",1
3,EMP-0179,2026-02-01,PR-2026-02,88641.00,2000.00,86641.00,2,"[PI-004052, PI-D00001]",1
4,EMP-0218,2025-05-01,PR-2025-05,178808.00,20865.08,157942.92,2,"[PI-000497, PI-D00008]",1
5,EMP-0250,2025-05-01,PR-2025-05,63282.00,2000.00,61282.00,2,"[PI-000514, PI-D00013]",1
6,EMP-0386,2026-03-01,PR-2026-03,57064.51,2000.00,55064.51,2,"[PI-004567, PI-D00004]",1
7,EMP-0408,2025-05-01,PR-2025-05,151737.00,14927.96,136809.04,2,"[PI-000603, PI-D00010]",1
8,EMP-0411,2025-07-01,PR-2025-07,112422.00,8162.83,104259.17,2,"[PI-001378, PI-D00012]",1
9,EMP-0430,2026-02-01,PR-2026-02,40306.00,2000.00,38306.00,2,"[PI-004184, PI-D00006]",1


In [53]:
alerts = payroll_features[
    payroll_features["combined_anomaly"]
].copy()

# Prepare payroll amounts
alerts["month"] = pd.to_datetime(alerts["month"])

calendar_days = alerts["month"].dt.days_in_month

expected_fixed_pay = (
    alerts["monthly_fixed_gross"]
    * alerts["payable_days"]
    / calendar_days
)

gross = alerts["gross_pay"].fillna(0)
net = alerts["net_pay"].fillna(0)

alerts["money_at_risk"] = 0.0
alerts["reason_code"] = ""
alerts["reason_text"] = ""

# 1. Negative net pay
mask = alerts["negative_net_pay"]

alerts.loc[mask, "reason_code"] = "NEGATIVE_NET_PAY"
alerts.loc[mask, "money_at_risk"] = net[mask].abs()

for idx in alerts.index[mask]:
    amount = alerts.at[idx, "money_at_risk"]
    alerts.at[idx, "reason_text"] = (
        f"Deductions exceed gross pay by ₹{amount:,.2f}, "
        "resulting in negative net pay."
    )

# 2. Zero net pay
mask = alerts["zero_net_pay"]

zero_risk = (
    expected_fixed_pay - gross
).clip(lower=0)

alerts.loc[mask, "reason_code"] = "ZERO_PAY"
alerts.loc[mask, "money_at_risk"] = zero_risk[mask]

for idx in alerts.index[mask]:
    amount = alerts.at[idx, "money_at_risk"]
    alerts.at[idx, "reason_text"] = (
        f"No net pay was recorded despite expected "
        f"prorated fixed pay of ₹{amount:,.2f}."
    )

# 3. Identify original and additional duplicate records
duplicate_rows = payroll_features[
    payroll_features["matching_payroll_record"]
].copy()

duplicate_keys = [
    "employee_id",
    "month",
    "run_id",
    "gross_pay",
    "total_deductions",
    "net_pay"
]

duplicate_partner = {}

for _, group in duplicate_rows.groupby(
    duplicate_keys, dropna=False
):
    ids = group["payroll_item_id"].astype(str).tolist()

    if len(ids) == 2:
        duplicate_partner[ids[0]] = ids[1]
        duplicate_partner[ids[1]] = ids[0]

# 4. Double-payment risk
mask = (
    alerts["matching_payroll_record"]
    & alerts["reason_code"].eq("")
)

is_additional = (
    alerts["payroll_item_id"]
    .astype(str)
    .str.startswith("PI-D")
)

duplicate_risk = net.clip(lower=0).where(is_additional, 0)

alerts.loc[mask, "reason_code"] = "DOUBLE_PAYMENT"
alerts.loc[mask, "money_at_risk"] = duplicate_risk[mask]

for idx in alerts.index[mask]:
    record_id = str(alerts.at[idx, "payroll_item_id"])
    partner = duplicate_partner.get(record_id, "matching record")
    amount = alerts.at[idx, "money_at_risk"]

    if record_id.startswith("PI-D"):
        reason = (
            f"Additional record matches original payment {partner}; "
            f"verify whether ₹{amount:,.2f} was paid twice."
        )
    else:
        reason = (
            f"This payment matches additional record {partner}; "
            "financial exposure is assigned to that record "
            "to avoid double-counting."
        )

    alerts.at[idx, "reason_text"] = reason

# 5. Isolation Forest statistical outliers
mask = (
    alerts["iforest_anomaly"]
    & alerts["reason_code"].eq("")
)

outlier_risk = (gross - expected_fixed_pay).abs()

alerts.loc[mask, "reason_code"] = "STATISTICAL_OUTLIER"
alerts.loc[mask, "money_at_risk"] = outlier_risk[mask]

for idx in alerts.index[mask]:
    amount = alerts.at[idx, "money_at_risk"]
    alerts.at[idx, "reason_text"] = (
        f"Gross pay differs from prorated fixed pay by "
        f"₹{amount:,.2f}; review whether overtime, variable "
        "pay or a salary change explains the difference."
    )

# Check that every alert has an explanation
assert alerts["reason_code"].ne("").all()
assert alerts["reason_text"].ne("").all()

print("Reason-code assignment completed!")
print("\nReason-code counts:")
display(alerts["reason_code"].value_counts())

print("\nMoney at risk by reason:")
display(
    alerts.groupby("reason_code")["money_at_risk"]
    .agg(["count", "sum", "mean"])
)

Reason-code assignment completed!

Reason-code counts:


reason_code
STATISTICAL_OUTLIER    176
DOUBLE_PAYMENT          30
NEGATIVE_NET_PAY         8
ZERO_PAY                 5
Name: count, dtype: int64


Money at risk by reason:


,count,sum,mean
reason_code,,,
DOUBLE_PAYMENT,30,1.412955e+06,47098.499000
NEGATIVE_NET_PAY,8,2.611747e+05,32646.838750
STATISTICAL_OUTLIER,176,4.022807e+07,228568.590815
ZERO_PAY,5,4.802423e+05,96048.453333


In [54]:
#creaeting 12 columns output 

In [55]:
# Convert Isolation Forest decision scores to 0–1
# Lower decision_function score = more anomalous

score_min = payroll_features["iforest_anomaly_score"].min()
score_max = payroll_features["iforest_anomaly_score"].max()

iforest_scores = (
    score_max - payroll_features["iforest_anomaly_score"]
) / (score_max - score_min)

# Build the official output
final_output = pd.DataFrame({
    "source_dataset": "payroll_monthly_features",
    "record_id": alerts["payroll_item_id"].astype(str),
    "employee_id": alerts["employee_id"].astype(str),
    "period": pd.to_datetime(alerts["month"]),
    "anomaly_score": iforest_scores.loc[alerts.index],
    "flagged": "Y",
    "reason_code": alerts["reason_code"],
    "reason_text": alerts["reason_text"],
    "money_at_risk": alerts["money_at_risk"],
    "model_version": "iforest-payroll-0.1+rules",
    "scored_at": pd.Timestamp.now().floor("s")
})

# Rank highest financial risk first.
# For equal amounts, use the anomaly score as a tie-breaker.
final_output = final_output.sort_values(
    ["money_at_risk", "anomaly_score"],
    ascending=[False, False],
    kind="stable"
).reset_index(drop=True)

# Generate unique result IDs after ranking
final_output.insert(
    0,
    "result_id",
    [f"AN-{i:04d}" for i in range(1, len(final_output) + 1)]
)

# Enforce exact official column order
official_columns = [
    "result_id",
    "source_dataset",
    "record_id",
    "employee_id",
    "period",
    "anomaly_score",
    "flagged",
    "reason_code",
    "reason_text",
    "money_at_risk",
    "model_version",
    "scored_at"
]

final_output = final_output[official_columns]

# Validation
assert final_output.shape == (219, 12)
assert final_output.columns.tolist() == official_columns
assert final_output["reason_code"].notna().all()
assert final_output["reason_text"].notna().all()
assert final_output["money_at_risk"].notna().all()
assert final_output["money_at_risk"].ge(0).all()
assert final_output["anomaly_score"].between(0, 1).all()

print("Official output created successfully!")
print("Shape:", final_output.shape)
print("Columns:", final_output.columns.tolist())

print("\nTotal estimated money at risk:")
print(f"₹{final_output['money_at_risk'].sum():,.2f}")

print("\nTop 20 highest-risk alerts:")
display(final_output.head(20))

Official output created successfully!
Shape: (219, 12)
Columns: ['result_id', 'source_dataset', 'record_id', 'employee_id', 'period', 'anomaly_score', 'flagged', 'reason_code', 'reason_text', 'money_at_risk', 'model_version', 'scored_at']

Total estimated money at risk:
₹42,382,443.93

Top 20 highest-risk alerts:


,result_id,source_dataset,record_id,employee_id,period,anomaly_score,flagged,reason_code,reason_text,money_at_risk,model_version,scored_at
0,AN-0001,payroll_monthly_features,PI-003304,EMP-0304,2025-12-01,0.821758,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,2.626596e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
1,AN-0002,payroll_monthly_features,PI-002595,EMP-0453,2025-10-01,0.820684,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,2.191879e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
2,AN-0003,payroll_monthly_features,PI-002124,EMP-0331,2025-09-01,0.806956,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.598202e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
3,AN-0004,payroll_monthly_features,PI-003441,EMP-0533,2025-12-01,0.780951,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.489401e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
4,AN-0005,payroll_monthly_features,PI-007199,EMP-0590,2026-09-01,0.749461,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.157625e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
5,AN-0006,payroll_monthly_features,PI-006419,EMP-0669,2026-07-01,0.749461,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.097901e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
6,AN-0007,payroll_monthly_features,PI-003796,EMP-0459,2026-01-01,0.774501,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,9.531630e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
7,AN-0008,payroll_monthly_features,PI-006595,EMP-0283,2026-08-01,0.765797,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,8.869770e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
8,AN-0009,payroll_monthly_features,PI-006837,EMP-0666,2026-08-01,0.762783,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,7.805970e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
9,AN-0010,payroll_monthly_features,PI-004312,EMP-0613,2026-02-01,0.772739,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,5.629095e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22


In [56]:
original_duplicate = (
    final_output["reason_code"].eq("DOUBLE_PAYMENT")
    & ~final_output["record_id"].str.startswith("PI-D")
)

for idx in final_output.index[original_duplicate]:
    record_id = final_output.at[idx, "record_id"]
    partner = duplicate_partner.get(record_id, "the additional record")

    final_output.at[idx, "reason_text"] = (
        f"This is the original payment and matches additional "
        f"record {partner}; duplicate exposure is counted once "
        "against that additional record."
    )

# Select five records from each reason code
review_parts = []

for code in [
    "STATISTICAL_OUTLIER",
    "DOUBLE_PAYMENT",
    "NEGATIVE_NET_PAY",
    "ZERO_PAY"
]:
    group = final_output[
        final_output["reason_code"] == code
    ]

    review_parts.append(
        group.nlargest(min(5, len(group)), "money_at_risk")
    )

review_sample = pd.concat(
    review_parts,
    ignore_index=True
)

# Rank the selected 20 by money at risk
review_sample = review_sample.sort_values(
    "money_at_risk",
    ascending=False
).reset_index(drop=True)

# Validate
assert len(review_sample) == 20
assert review_sample["reason_code"].value_counts().eq(5).all()
assert final_output["money_at_risk"].is_monotonic_decreasing
assert final_output.columns.tolist() == official_columns

# Save evidence
reports_dir = Path("/workspace/reports")
reports_dir.mkdir(parents=True, exist_ok=True)

final_output.to_csv(
    reports_dir / "task7_payroll_anomaly_results.csv",
    index=False
)

final_output.to_excel(
    reports_dir / "task7_payroll_anomaly_results.xlsx",
    index=False
)

review_sample.to_excel(
    reports_dir / "task7_payroll_20_row_review.xlsx",
    index=False
)

print("Task 7 output saved successfully!")
print("Full output:", len(final_output), "records")
print("Human review sample:", len(review_sample), "records")

print("\nReview sample by reason code:")
display(review_sample["reason_code"].value_counts())

print("\n20-row review table:")
display(review_sample)

Task 7 output saved successfully!
Full output: 219 records
Human review sample: 20 records

Review sample by reason code:


reason_code
STATISTICAL_OUTLIER    5
DOUBLE_PAYMENT         5
ZERO_PAY               5
NEGATIVE_NET_PAY       5
Name: count, dtype: int64


20-row review table:


,result_id,source_dataset,record_id,employee_id,period,anomaly_score,flagged,reason_code,reason_text,money_at_risk,model_version,scored_at
0,AN-0001,payroll_monthly_features,PI-003304,EMP-0304,2025-12-01,0.821758,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,2.626596e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
1,AN-0002,payroll_monthly_features,PI-002595,EMP-0453,2025-10-01,0.820684,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,2.191879e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
2,AN-0003,payroll_monthly_features,PI-002124,EMP-0331,2025-09-01,0.806956,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.598202e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
3,AN-0004,payroll_monthly_features,PI-003441,EMP-0533,2025-12-01,0.780951,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.489401e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
4,AN-0005,payroll_monthly_features,PI-007199,EMP-0590,2026-09-01,0.749461,Y,STATISTICAL_OUTLIER,Gross pay differs from prorated fixed pay by ₹...,1.157625e+06,iforest-payroll-0.1+rules,2026-10-31 06:10:22
5,AN-0036,payroll_monthly_features,PI-D00002,EMP-0112,2025-10-01,0.348803,Y,DOUBLE_PAYMENT,Additional record matches original payment PI-...,2.721235e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
6,AN-0075,payroll_monthly_features,PI-005048,EMP-0503,2026-04-01,0.512790,Y,ZERO_PAY,No net pay was recorded despite expected prora...,1.951043e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
7,AN-0091,payroll_monthly_features,PI-005869,EMP-0503,2026-06-01,1.000000,Y,NEGATIVE_NET_PAY,"Deductions exceed gross pay by ₹174,244.25, re...",1.742442e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
8,AN-0104,payroll_monthly_features,PI-D00008,EMP-0218,2025-05-01,0.175458,Y,DOUBLE_PAYMENT,Additional record matches original payment PI-...,1.579429e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22
9,AN-0114,payroll_monthly_features,PI-004865,EMP-0142,2026-04-01,0.512790,Y,ZERO_PAY,No net pay was recorded despite expected prora...,1.393180e+05,iforest-payroll-0.1+rules,2026-10-31 06:10:22


In [57]:
#export the model 

In [58]:
import joblib
from pathlib import Path

# Create models directory
models_path = Path("/workspace/models")
models_path.mkdir(exist_ok=True)

# Package the reason-code definitions and validated rules
reason_code_artifact = {
    "payroll": {
        "reason_definitions": reason_definitions,
        "priority": [
            "negative_net_pay",
            "zero_net_pay",
            "matching_payroll_record",
            "iforest_anomaly"
        ],
        "duplicate_keys": [
            "employee_id",
            "month",
            "run_id",
            "gross_pay",
            "total_deductions",
            "net_pay"
        ]
    },

    "attendance": {
        "priority": [
            "NO_ATTENDANCE",
            "LONG_UNPAID_ABSENCE",
            "FREQUENT_LATE",
            "PATTERN_SHIFT"
        ],
        "rules": {
            "NO_ATTENDANCE": {
                "field": "attendance_rate",
                "operator": "==",
                "value": 0
            },
            "LONG_UNPAID_ABSENCE": {
                "field": "longest_unpaid_streak_days",
                "operator": ">=",
                "value": 7
            },
            "FREQUENT_LATE": {
                "field": "late_arrivals",
                "operator": ">=",
                "value": 14
            },
            "PATTERN_SHIFT": {
                "default": True
            }
        }
    }
}

# Export
joblib.dump(
    reason_code_artifact,
    models_path / "reason_codes.jo|blib"
)

print("Task 7 reason-code artifact exported successfully!")
print("File:", models_path / "reason_codes.joblib")

Task 7 reason-code artifact exported successfully!
File: /workspace/models/reason_codes.joblib
